# Module 4 Homework Solution — Stock Analytics 2026
**Author:** Quant Developer  
**Date:** 2026-10-29  
**Environment:** Python 3, pandas 3.0.5, scikit-learn 1.9.1, numpy 2.5.2  
**Data:**  (229,932 rows × 203 columns, 33 tickers, 2000–2024)
---

## Table of Contents
1. [Setup and Data Loading](#setup)
2. [Question 1: Less Features is More](#q1)
3. [Question 2: Most Efficient Random Forest Decision Rule](#q2)
4. [Question 3: Redefining the Growth Target](#q3)
5. [Question 4: Ideal Trading Strategy](#q4)
6. [Bubble Chart Visualization](#bubble)


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load data
df = pd.read_parquet('c:/stock analytics 2026/data.parquet')
print(f'Data shape: {df.shape}')
print(f'Tickers: {df["Ticker"].nunique()}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')


In [ ]:
# Feature definitions (Q1: removed ticker and month_wom from CATEGORICAL)
CATEGORICAL = ['month', 'weekday', 'ticker_type']
GROWTH = [g for g in df.columns if (g.find('growth_')==0)&(g.find('future')<0)]
TECH_IND = ['adx','adxr','apo','aroon_1','aroon_2','aroonosc','bop','cci','cmo','dx',
 'macd','macdsignal','macdhist','macd_ext','macdsignal_ext','macdhist_ext',
 'macd_fix','macdsignal_fix','macdhist_fix','mfi','minus_di','mom','plus_di','dm',
 'ppo','roc','rocp','rocr','rocr100','rsi','slowk','slowd','fastk','fastd',
 'fastk_rsi','fastd_rsi','trix','ultosc','willr','ad','adosc','obv','atr','natr',
 'ht_dcperiod','ht_dcphase','ht_phasor_inphase','ht_phasor_quadrature','ht_sine_sine',
 'ht_sine_leadsine','ht_trendmod','avgprice','medprice','typprice','wclprice']
TECH_PAT = [g for g in df.columns if g.find('cdl')>=0]
CUSTOM_NUM = ['SMA10','SMA20','growing_moving_average','high_minus_low_relative','volatility','ln_volume']
MACRO = ['gdppot_us_yoy','gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','FEDFUNDS','DGS1','DGS5','DGS10']
NUMERICAL = GROWTH + TECH_IND + TECH_PAT + CUSTOM_NUM + MACRO
print(f'NUMERICAL count: {len(NUMERICAL)}')
print(f'CATEGORICAL: {CATEGORICAL}')


In [ ]:
# Generate dummies
df.loc[:,'Month_str'] = df.Month.dt.strftime('%B')
df.loc[:,'Weekday_str'] = df.Weekday.astype(str)
CATEGORICAL_STR = ['Month_str', 'Weekday_str', 'ticker_type']
dummy_variables = pd.get_dummies(df[CATEGORICAL_STR], dtype='int32')
DUMMIES = dummy_variables.keys().to_list()
df_with_dummies = pd.concat([df, dummy_variables], axis=1)
print(f'Dummy count: {len(DUMMIES)}')
print(f'Total features: {len(NUMERICAL) + len(DUMMIES)}')
# Q1 check
feat_count = len(NUMERICAL) + len(DUMMIES)
print(f'\nQ1: Feature count = {feat_count} (expected ~208, 31% fewer than 301)')


In [ ]:
# Target variable
df_with_dummies['is_positive_growth_30d_future_orig'] = (df_with_dummies['growth_future_30d'] > 1).astype(int)
print(f'Original target dist: {df_with_dummies["is_positive_growth_30d_future_orig"].value_counts(normalize=True).to_dict()}')
print(f'growth_future_30d median: {df_with_dummies["growth_future_30d"].median():.4f}')
print(f'growth_future_30d 75th pct: {df_with_dummies["growth_future_30d"].quantile(0.75):.4f}')


In [ ]:
new_df = df_with_dummies.copy()
pd.options.mode.chained_assignment = None
new_df.replace([np.inf, -np.inf], np.nan, inplace=True)
new_df.fillna(0, inplace=True)
features_list = NUMERICAL + DUMMIES
to_predict = 'is_positive_growth_30d_future_orig'
train_df = new_df[new_df.split.isin(['train','validation'])].copy(deep=True)
test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)
X_train = train_df[features_list + [to_predict, 'Date', 'Ticker']]
X_test = test_df[features_list + [to_predict, 'Date', 'Ticker']]
y_train = X_train[to_predict]
y_test = X_test[to_predict]
X_train_f = X_train.drop([to_predict, 'Date', 'Ticker'], axis=1)
X_test_f = X_test.drop([to_predict, 'Date', 'Ticker'], axis=1)
clf_best = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf_best.fit(X_train_f, y_train)
rf_best = RandomForestClassifier(max_depth=13, n_estimators=100, random_state=42, n_jobs=-1)
rf_best.fit(X_train_f, y_train)
clf_10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf_10.fit(X_train_f, y_train)
clf_20 = DecisionTreeClassifier(max_depth=20, random_state=42)
clf_20.fit(X_train_f, y_train)
print('Models trained successfully')
